# 03 · Factor Structure
PCA loadings, Kalman innovation norm, factor behaviour during COVID.
Adjust parameters with the widgets below.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from data.synthetic import generate_em_universe
from modules.pca_kalman import compute_dynamic_factors_v2
print('Imports OK')


### Parameters (interactive)


In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_start  = widgets.DatePicker(description='Start date',
               value=__import__('datetime').date(2015, 1, 1))
w_end    = widgets.DatePicker(description='End date',
               value=__import__('datetime').date(2024, 12, 31))
w_window = widgets.IntSlider(min=63, max=504, step=21, value=252,
               description='Window', style={'description_width':'initial'})
w_slow_w = widgets.IntSlider(min=126, max=756, step=63, value=504,
               description='Slow window', style={'description_width':'initial'})
w_lam    = widgets.FloatSlider(min=0.90, max=0.99, step=0.01, value=0.94,
               description='EWMA lambda', style={'description_width':'initial'},
               readout_format='.2f')
w_vs     = widgets.Checkbox(value=True, description='Vol-standardize turbulence')

display(widgets.VBox([
    widgets.HBox([w_start, w_end]),
    w_window, w_slow_w, w_lam, w_vs,
]))
print('Adjust parameters above, then run the next cell.')


In [ ]:
# Read current widget values — run this after adjusting sliders
START   = str(w_start.value)
END     = str(w_end.value)
WINDOW  = w_window.value
SLOW_W  = w_slow_w.value
LAM     = w_lam.value
VOL_STD = w_vs.value

print(f'Parameters: start={START}  end={END}  window={WINDOW}')
print(f'            slow_w={SLOW_W}  lam={LAM:.2f}  vol_std={VOL_STD}')


### Data & Computation


In [ ]:
uni   = generate_em_universe(seed=42, start=START)
panel = uni.panel
N_COMP = 3

dyn = compute_dynamic_factors_v2(
    panel, window=WINDOW, n_components=N_COMP, min_periods=60, lam=LAM)

print(f'Panel  : {panel.shape}')
print(f'Factors: {dyn.pca.factor_scores.shape}')
print(f'innovation_norm non-null: {dyn.pca.innovation_norm.notna().sum()}')


## 2 · Rolling Explained Variance by Factor


In [ ]:
scores    = dyn.pca.factor_scores.dropna()
roll_var  = scores.rolling(WINDOW, min_periods=60).var()
roll_tot  = roll_var.sum(axis=1)
evr       = roll_var.div(roll_tot, axis=0).clip(0, 1)
fcolors   = ['#e74c3c','#3498db','#2ecc71']

fig, ax = plt.subplots(figsize=(13, 4))
bottom = np.zeros(len(evr))
for i, col in enumerate(evr.columns):
    ax.fill_between(evr.index, bottom, bottom + evr[col].fillna(0),
                    alpha=0.8, color=fcolors[i], label=col)
    bottom = bottom + evr[col].fillna(0).values
ax.set_ylabel(f'Rolling var share (window={WINDOW}d)')
ax.set_title('Rolling Explained Variance by Factor')
ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()


## 3 · Latest Factor Loadings Heatmap
> SVD sign is arbitrary — interpret the pattern, not the sign.


In [ ]:
common_idx = scores.index.intersection(panel.index)
load_mat   = pd.DataFrame(
    {col: panel.loc[common_idx].corrwith(scores.loc[common_idx][col])
     for col in scores.columns})

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(load_mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.3, ax=ax, annot_kws={'size': 9})
ax.set_title('Factor Loadings (full-sample corr of scores with returns)')
ax.set_xlabel('Factor'); ax.set_ylabel('Asset')
plt.tight_layout(); plt.show()


## 4 · F1 Loading Evolution Per Asset Over Time


In [ ]:
ROLL = 63
f1 = scores.iloc[:, 0]
acolors = ['#00d4aa','#e74c3c','#f39c12','#9b59b6','#3498db',
           '#1abc9c','#e67e22','#8e44ad','#2980b9','#27ae60']

fig, ax = plt.subplots(figsize=(13, 5))
for i, asset in enumerate(panel.columns):
    common = f1.index.intersection(panel[asset].dropna().index)
    rc = f1.loc[common].rolling(ROLL, min_periods=30).corr(panel[asset].loc[common])
    ax.plot(rc.index, rc, label=asset, color=acolors[i % len(acolors)], lw=1.0, alpha=0.85)
ax.axhline(0, color='white', lw=0.6, alpha=0.3)
ax.set_ylabel(f'Rolling {ROLL}d corr with F1')
ax.set_title('F1 Loading Evolution')
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()


## 5 · Kalman Innovation Norm  ||delta R||_F


In [ ]:
inorm = dyn.pca.innovation_norm.dropna()
mu_i  = float(inorm.mean()); sd_i = float(inorm.std())
thr   = mu_i + 2 * sd_i

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(inorm.index, inorm.values, color='#e74c3c', lw=0.9)
ax.fill_between(inorm.index, inorm.values, alpha=0.15, color='#e74c3c')
ax.axhline(thr,  color='#f39c12', ls='--', lw=1.0, label=f'mean+2sd={thr:.3f}')
ax.axhline(mu_i, color='white',   ls=':',  lw=0.6, alpha=0.4)
ax.set_ylabel('||delta R||_F'); ax.set_title('Correlation Regime Shock')
ax.legend(fontsize=9); ax.grid(alpha=0.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()


## 6 · Innovation Spike Table


In [ ]:
spikes = inorm[inorm > thr].reset_index()
spikes.columns = ['Date','||dR||_F']
spikes['Date'] = spikes['Date'].dt.strftime('%Y-%m-%d')
spikes['||dR||_F'] = spikes['||dR||_F'].round(4)
print(f'Threshold (mean+2sd): {thr:.4f}   N spikes: {len(spikes)}')
print(spikes.to_string(index=False))


## 7 · Factor Scores During COVID (2020-01-01 to 2020-06-30)


In [ ]:
sc_covid = scores.loc['2020-01-01':'2020-06-30']
fig, axes = plt.subplots(N_COMP, 1, figsize=(13, 2.5*N_COMP), sharex=True)
for i, ax in enumerate(axes):
    col = sc_covid.columns[i]
    ax.plot(sc_covid.index, sc_covid[col], color=fcolors[i], lw=1.2)
    ax.axhline(0, color='white', lw=0.5, alpha=0.3)
    ax.set_ylabel(col); ax.grid(alpha=0.15); ax.set_title(f'{col} — COVID period')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout(); plt.show()


## 8 · Export to Excel


In [ ]:
from datetime import date
os.makedirs('../exports', exist_ok=True)
filename = f'../exports/03_factor_structure_{date.today()}.xlsx'
with pd.ExcelWriter(filename, engine='openpyxl') as writer:
    scores.to_excel(writer, sheet_name='Factor Scores')
    dyn.to_frame().to_excel(writer, sheet_name='Kalman Filtered')
    inorm.to_frame().to_excel(writer, sheet_name='Innovation Norm')
    load_mat.to_excel(writer, sheet_name='Latest Loadings')
    spikes.to_excel(writer, sheet_name='Innovation Spikes', index=False)
    sc_covid.to_excel(writer, sheet_name='COVID Episode')
print(f'Exported: {filename}')
